# Exploration 07 — quantum cost, finite shots and hardware noise (M5, M6 cost table; simulators only)

> **Exploratory** (CLAUDE.md, "Exploration"). **Inputs:** Z and R only. **Data:** training sets S_(s,N) and the
> development sets `dev` and `dev_unseen`; no test set is read. **Quantum:** local simulators only. The noise models
> come from `qiskit_ibm_runtime.fake_provider` (calibration snapshots of real IBM devices, run on Aer on this
> computer). **Nothing is sent to IBM hardware.**

Notebook 06 compared the quantum models with exact statevectors: the numbers a perfect quantum computer would give
with infinitely many measurements. A real device gives neither. This notebook asks what the quantum models would
**cost** to run, and how much accuracy **finite shots** and **hardware noise** take away.

Concepts, briefly:
- **Shots.** Each circuit is run S times and every qubit measured each time; a probability is estimated as a count
  over S, with standard error √(p(1−p)/S). A fidelity-kernel entry k(x, x′) is the probability of reading all zeros
  after running U(x′) then U(x)†. An ideal device's estimate is exactly Binomial(S, k)/S, so the fast path draws
  that from the exact kernel; section 2 checks it against Aer running the real circuits.
- **Transpilation.** Hardware runs only its native gates (on IBM Heron: rz, sx, x and the two-qubit cz) between
  physically connected qubits. Qiskit rewrites the circuit accordingly, which adds gates; two-qubit gates are the
  main error source.
- **Noise.** `AerSimulator` with the noise model of `FakeFez` (Heron r2, 156 qubits) applies that device's calibrated
  gate errors, decoherence and readout errors. A fidelity kernel suffers in a specific way: noise pushes states
  toward the maximally mixed state, so every overlap drifts toward 2⁻ⁿ.

In [ ]:
# Environment check: stops here, with setup instructions, unless this kernel is the project
# environment (README.md, "Setup"). Version differences from requirements.txt only warn.
try:
    from qm9dipole.provenance import check_environment
except ModuleNotFoundError as e:
    raise RuntimeError(
        "The qm9dipole package is not installed in this kernel's environment. Follow "
        "README.md (Setup), then select that environment as the notebook kernel."
    ) from e

print(check_environment())

In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge

from qm9dipole import REPO_ROOT
from qm9dipole.cost import circuits_needed, gate_counts, qpu_seconds, transpiled_costs
from qm9dipole.evaluate import clip_predictions, scores
from qm9dipole.explore import feature_sets, load_catalog, load_features, load_preprocessing
from qm9dipole.models.quantum import build_encoding_circuit
from qm9dipole.models.quantum_kernel import (FidelityKernel, KernelRidgeFitter, ProjectedKernel, default_qubits,
                                             quantum_ridge_build)
from qm9dipole.noise import (basis_circuits, depolarizing_mitigation, overlap_circuit, repair_kernel, sample_bloch,
                             sample_overlaps, shot_expectations, shot_kernel)
from qm9dipole.plots import AXIS, MODEL_COLORS, MODEL_LABELS, MUTED, SEQUENTIAL, SURFACE, model_style, use_style
from qm9dipole.provenance import RESULTS_DIR, save_result
from qm9dipole.splits import load_splits

# Development switch: QM9DIPOLE_SMOKE=1 runs one seed at N = 100 with small subsets. Unset for real runs.
SMOKE = os.environ.get("QM9DIPOLE_SMOKE") == "1"

use_style()
FIG_DIR = REPO_ROOT / "figures"
BACKEND = "FakeFez"  # IBM Heron r2 calibration snapshot; simulated locally on Aer
features = load_features()
SETS = feature_sets(load_catalog())
PRE = load_preprocessing()
SC_B, TG_B = PRE["scaling"]["track_b"], PRE["target"]["track_b"]
RED = (PRE["reduction"]["method"], int(PRE["reduction"]["k"]))
splits = load_splits()
y = features["mu"]
EVAL = {"dev": splits.dev, "dev_unseen": splits.dev_unseen}
SIZES = [100] if SMOKE else list(splits.track_b_sizes)
SEEDS = [0] if SMOKE else list(splits.train)
NMAX = max(SIZES)
XA = features[SETS["all_legal"]]
main6 = pd.read_csv(RESULTS_DIR / "explore06_main.csv")
HEAD = json.loads((RESULTS_DIR / "explore06_headline.meta.json").read_text())["head"]
FID = HEAD if HEAD.startswith("qkrr") else "qkrr"  # the fidelity-kernel model studied here
KERNELS = {"qkrr": ("zz", 2), "qkrr_ry": ("ry", 2), "qkrr_ryrz": ("ry_rz", 2), "qkrr_product": ("ry", 1), "pqk": ("zz", 2)}
print(f"headline quantum model of notebook 06: {MODEL_LABELS[HEAD]}; fidelity model studied: {MODEL_LABELS[FID]}")


def chosen(model, seed, n):
    row = main6[(main6["model"] == model) & (main6["seed"] == seed) & (main6["n_train"] == n) & (main6["eval_set"] == "dev")]
    return json.loads(row["details"].iloc[0])["params"], row


def fixed_fit(model, seed, n):
    """Refit a notebook-06 quantum kernel model at its chosen settings (one candidate, no search)."""
    p, _ = chosen(model, seed, n)
    enc, reps = KERNELS[model]
    kern = (ProjectedKernel(enc, reps, gammas=[p["gamma"]], gammas_p=[p["gamma_p"]]) if model == "pqk"
            else FidelityKernel(enc, reps, gammas=[p["gamma"]]))
    ids = splits.train[seed][n]
    return KernelRidgeFitter(XA, kern, seed, scaling=SC_B, target=TG_B, reduction=RED, alphas=(p["alpha"],),
                             cv_max_n=NMAX).fit(ids, y.loc[ids])

## 1. What would it cost on IBM hardware?

Every circuit at k = 4 … 16 inputs for the three encoders: the encoding U(x) with a measurement (what the projected
kernel and the team's ⟨Z⟩ model run, once per basis), and the overlap circuit C(x, x′) = U(x′)·U(x)† that one
fidelity-kernel entry needs. Logical counts first, then after transpiling for `FakeFez`.

In [ ]:
K_LIST = [4, 10] if SMOKE else [4, 6, 8, 10, 12, 16]
rows = []
for enc, reps in [("zz", 2), ("ry", 2), ("ry_rz", 2)]:
    for k in K_LIST:
        U = build_encoding_circuit(k, default_qubits(enc, k), encoding=enc, reps=reps)
        for kind, circ in (("U(x) + measure", basis_circuits(U)["Z"]), ("overlap C(x, x′)", overlap_circuit(U))):
            log, tr = gate_counts(circ), transpiled_costs(circ, BACKEND)
            rows.append({"encoding": enc, "inputs k": k, "circuit": kind, "qubits": log["qubits"],
                         "logical depth": log["depth"], "logical 2q gates": log["two_qubit_gates"],
                         "transpiled depth": tr["depth"], "transpiled 2q gates (cz)": tr["two_qubit_gates"],
                         "transpiled 2q depth": tr["two_qubit_depth"], "duration (µs)": 1e6 * tr["duration_s"]})
circuit_costs = pd.DataFrame(rows)
_ = save_result(circuit_costs, "explore07_circuit_costs", backend=BACKEND, optimization_level=2,
                label="logical and transpiled circuit costs of the team encoders (local transpilation; no hardware)")
display(circuit_costs.set_index(["encoding", "inputs k", "circuit"]).style.format({"duration (µs)": "{:.1f}"})
        .set_caption(f"circuit costs, transpiled for {BACKEND} (optimization level 2)"))

**Circuits and QPU time per model**, for training on N molecules and predicting the 9,183 development molecules, at
1,000 shots per circuit, with each model's circuit at k = 10 (the ZZ map unless stated). QPU time ≈ circuits × shots
× (circuit duration + ~2 µs readout + ~250 µs repetition delay): a lower bound that ignores job overheads. For scale,
IBM's Open Plan gives about 10 minutes of QPU time per 28 days.

In [ ]:
SHOTS_COST = 1000
n_dev = len(splits.dev) + len(splits.dev_unseen)
dur = circuit_costs[(circuit_costs["inputs k"] == RED[1]) & (circuit_costs["encoding"] == KERNELS[FID][0])].set_index("circuit")["duration (µs)"] * 1e-6
dur_pqk = circuit_costs[(circuit_costs["inputs k"] == RED[1]) & (circuit_costs["encoding"] == "zz")].set_index("circuit")["duration (µs)"] * 1e-6
rows = []
for n in (100, 300, 1000):
    for model, kind, d in ((FID, "fidelity", dur["overlap C(x, x′)"]), ("pqk", "projected", dur_pqk["U(x) + measure"]),
                           ("qridge", "z_readout", dur_pqk["U(x) + measure"])):
        train, predict = circuits_needed(kind, n, n_dev)
        sec_train, sec_pred = qpu_seconds(train, SHOTS_COST, d), qpu_seconds(predict, SHOTS_COST, d)
        rows.append({"model": MODEL_LABELS[model], "N": n, "circuits to train": train,
                     "circuits to predict 9,183 molecules": predict, "QPU hours to train": sec_train / 3600,
                     "QPU hours to predict": sec_pred / 3600,
                     "Open Plan months (train + predict)": (sec_train + sec_pred) / 600})
model_costs = pd.DataFrame(rows)
_ = save_result(model_costs, "explore07_model_costs", shots=SHOTS_COST, backend=BACKEND, rep_delay_s=250e-6,
                label="circuits and rough QPU time (lower bound) per quantum model")
display(model_costs.set_index(["model", "N"]).style.format({"QPU hours to train": "{:.3g}", "QPU hours to predict": "{:.3g}",
        "Open Plan months (train + predict)": "{:,.1f}", "circuits to train": "{:,}", "circuits to predict 9,183 molecules": "{:,}"})
        .set_caption(f"cost per model at {SHOTS_COST:,} shots per circuit"))

fig, ax = plt.subplots(figsize=(7.5, 4.3))
for model in (FID, "pqk", "qridge"):
    s = model_costs[model_costs["model"] == MODEL_LABELS[model]]
    ax.plot(s["N"], s["QPU hours to train"] + s["QPU hours to predict"], label=MODEL_LABELS[model], **model_style(model))
ax.axhline(10 / 60, color=MUTED, linewidth=1, linestyle=":")
ax.annotate("Open Plan: 10 min per 28 days", (100, 10 / 60), xytext=(0, 4), textcoords="offset points", fontsize=8, color=MUTED)
ax.set(xscale="log", yscale="log", xlabel="training labels N", ylabel="QPU hours (lower bound)",
       title="QPU time to train and predict the development sets")
ax.set_xticks([100, 300, 1000], ["100", "300", "1000"])
ax.legend(fontsize=8.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "explore07_qpu_time.png")
plt.show()

## 2. Checking the shot model against real circuits (ideal Aer)

The fast path draws Binomial(S, k)/S from exact kernel values. Here 60 actual overlap circuits (development molecule i
against training molecule i of S_(0,N), at the chosen γ) run on Aer's ideal simulator with 2,000 shots, and the
z-scores (estimate − exact)/√(k(1−k)/S) should look standard normal. The same for projected-kernel Bloch vectors.

In [ ]:
SH = 2000
fit0 = fixed_fit(FID, 0, NMAX)
g0 = fit0.embed_setting_["gamma"]
tr0 = splits.train[0][NMAX]
m = 20 if SMOKE else 60
A = g0 * fit0.transform_inputs(XA.loc[splits.dev[:m]].to_numpy())
B = g0 * fit0.transform_inputs(XA.loc[tr0[:m]].to_numpy())
exact = np.diag(fit0.gram_to_train(XA.loc[splits.dev[:m]].to_numpy())[:, :m])  # dev molecule i vs training molecule i
est = sample_overlaps(fit0.kernel.circuit(RED[1]), A, B, SH, seed=1, n_jobs=-1, chunk=10)
zs = (est - exact) / np.sqrt(np.clip(exact * (1 - exact), 1.0 / SH, None) / SH)
print(f"fidelity kernel, {m} circuits: exact k from {exact.min():.3f} to {exact.max():.3f}; "
      f"z-scores mean {zs.mean():+.2f}, sd {zs.std():.2f}, max |z| {np.abs(zs).max():.2f}")
fitp0 = fixed_fit("pqk", 0, NMAX)
Bl_exact = fitp0.embed_X(XA.loc[splits.dev[:10]].to_numpy())
Ap = fitp0.embed_setting_["gamma"] * fitp0.transform_inputs(XA.loc[splits.dev[:10]].to_numpy())
Bl_est = sample_bloch(fitp0.kernel.circuit(RED[1]), Ap, SH, seed=2)
zb = (Bl_est - Bl_exact) / np.sqrt(np.clip(1 - Bl_exact**2, 1.0 / SH, None) / SH)
print(f"projected kernel, 10 molecules × {Bl_exact.shape[1]} Bloch coordinates: z mean {zb.mean():+.2f}, sd {zb.std():.2f}, "
      f"max |z| {np.abs(zb).max():.2f}")
validation = pd.DataFrame({"what": ["fidelity entries"] * len(zs) + ["Bloch coordinates"] * zb.size,
                           "z": np.r_[zs, zb.ravel()]})
_ = save_result(validation, "explore07_shot_validation", shots=SH, label="z-scores of ideal Aer sampling against exact values")

## 3. Finite shots on the full development sets

For every seed and N, at the settings notebook 06 chose:
- **fidelity kernel, test kernel from shots:** the training kernel exact (it could be measured once, with many shots),
  every development-molecule entry estimated with S shots (PLAN §7.5 default);
- **fidelity kernel, both from shots:** the training kernel estimated too (then symmetrized, unit diagonal, negative
  eigenvalues clipped) and the model refit;
- **projected kernel:** every Bloch coordinate of every molecule, training and development, from S shots per basis;
  model refit;
- **team quantum-feature ridge:** every ⟨Z⟩ from S shots, training and development; ridge refit.
S ∈ {100, 1,000, 10,000}, 5 repetitions each.

In [ ]:
SHOTS = [100, 1000, 10000]
REPS = 2 if SMOKE else 5
rows = []


def add(model, mode, seed, n, shots, rep, ev, pred):
    rows.append({"model": model, "mode": mode, "seed": seed, "n_train": n, "shots": shots, "rep": rep, "eval_set": ev,
                 **scores(y.loc[EVAL[ev]].to_numpy(), pred)})


start = time.perf_counter()
for seed in SEEDS:
    for n in SIZES:
        ids = splits.train[seed][n]
        Xev = {ev: XA.loc[i].to_numpy() for ev, i in EVAL.items()}
        # Fidelity kernel
        fit = fixed_fit(FID, seed, n)
        Ktr = fit.kernel.gram(fit.train_embedding_, fit.train_embedding_, {})
        Kev = {ev: fit.gram_to_train(X) for ev, X in Xev.items()}
        for ev in EVAL:
            pred = fit.predict_from_gram(Kev[ev])
            ref = main6[(main6["model"] == FID) & (main6["seed"] == seed) & (main6["n_train"] == n) & (main6["eval_set"] == ev)]["mae_D"].iloc[0]
            assert abs(np.abs(pred - y.loc[EVAL[ev]].to_numpy()).mean() - ref) < 1e-9, "refit differs from notebook 06"
            add(FID, "exact", seed, n, np.inf, 0, ev, pred)
        for S in SHOTS:
            for r in range(REPS):
                rng = np.random.default_rng([seed, n, S, r])
                noisy_model = KernelRidge(kernel="precomputed", alpha=fit.alpha_).fit(repair_kernel(shot_kernel(Ktr, S, rng)), fit.z_train_)
                for ev in EVAL:
                    Khat = shot_kernel(Kev[ev], S, rng)
                    add(FID, "test kernel from shots", seed, n, S, r, ev, fit.predict_from_gram(Khat))
                    add(FID, "training and test kernels from shots", seed, n, S, r, ev, fit.predict_from_gram(Khat, noisy_model))
        # Projected kernel
        fitp = fixed_fit("pqk", seed, n)
        Btr = fitp.train_embedding_
        Bev = {ev: fitp.embed_X(X) for ev, X in Xev.items()}
        for ev in EVAL:
            add("pqk", "exact", seed, n, np.inf, 0, ev, fitp.predict_from_gram(fitp.kernel.gram(Bev[ev], Btr, fitp.kernel_setting_)))
        for S in SHOTS:
            for r in range(REPS):
                rng = np.random.default_rng([seed, n, S, r, 1])
                Bh = shot_expectations(Btr, S, rng)
                mdl = KernelRidge(kernel="precomputed", alpha=fitp.alpha_).fit(fitp.kernel.gram(Bh, Bh, fitp.kernel_setting_), fitp.z_train_)
                for ev in EVAL:
                    Kh = fitp.kernel.gram(shot_expectations(Bev[ev], S, rng), Bh, fitp.kernel_setting_)
                    add("pqk", "features from shots", seed, n, S, r, ev, fitp.predict_from_gram(Kh, mdl))
        # The team's quantum-feature ridge
        p, _ = chosen("qridge", seed, n)
        est, _ = quantum_ridge_build(seed, SC_B, TG_B, RED, XA.shape[1])
        est.set_params(**p).fit(XA.loc[ids].to_numpy(), y.loc[ids].to_numpy())
        prefix, qr = est.regressor_[:-1], est.regressor_[-1]
        Ftr = qr.quantum_features(prefix.transform(XA.loc[ids].to_numpy()))
        Fev = {ev: qr.quantum_features(prefix.transform(X)) for ev, X in Xev.items()}
        z_tr = est.transformer_.transform(y.loc[ids].to_numpy()[:, None]).ravel()
        inv = lambda z: clip_predictions(est.transformer_.inverse_transform(z[:, None]).ravel())
        for ev in EVAL:
            add("qridge", "exact", seed, n, np.inf, 0, ev, clip_predictions(est.predict(Xev[ev])))
        for S in SHOTS:
            for r in range(REPS):
                rng = np.random.default_rng([seed, n, S, r, 2])
                rdg = Ridge(alpha=p["regressor__model__alpha"]).fit(shot_expectations(Ftr, S, rng), z_tr)
                for ev in EVAL:
                    add("qridge", "features from shots", seed, n, S, r, ev, inv(rdg.predict(shot_expectations(Fev[ev], S, rng))))
    print(f"seed {seed}: {(time.perf_counter() - start) / 60:.1f} min", flush=True)
shots = pd.DataFrame(rows)
_ = save_result(shots, "explore07_shots", shots=SHOTS, reps=REPS, smoke=SMOKE,
                label="exploratory; finite-shot inference (exact binomial sampling) at notebook 06's chosen settings; dev scores")

In [ ]:
agg = shots.groupby(["model", "mode", "eval_set", "n_train", "shots"])[["mae_D", "rmse_D"]].agg(["mean", "std"])
t = agg.xs("dev", level="eval_set")["mae_D"]
t = (t["mean"].map("{:.3f}".format) + " ± " + t["std"].fillna(0).map("{:.3f}".format)).unstack("shots")
t.columns = ["exact" if np.isinf(c) else f"S = {int(c):,}" for c in t.columns]
display(t.style.set_caption("dev MAE (D) vs shots per circuit (mean ± sd over seeds and repetitions)"))
tu = agg.xs("dev_unseen", level="eval_set")["mae_D"]["mean"].unstack("shots")
tu.columns = ["exact" if np.isinf(c) else f"S = {int(c):,}" for c in tu.columns]
display(tu.style.format("{:.3f}").set_caption("dev_unseen MAE (D) vs shots (mean)"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
modes = [(FID, "test kernel from shots", "-"), (FID, "training and test kernels from shots", ":"),
         ("pqk", "features from shots", "--"), ("qridge", "features from shots", "-.")]
for ax, ev in zip(axes, EVAL):
    s = shots[(shots["eval_set"] == ev) & (shots["n_train"] == NMAX)]
    for model, mode, ls in modes:
        v = s[(s["model"] == model) & (s["mode"] == mode)].groupby("shots")["mae_D"].agg(["mean", "std"])
        st = {**model_style(model), "linestyle": ls}
        ax.errorbar(v.index, v["mean"], yerr=v["std"], capsize=2.5, label=f"{MODEL_LABELS[model]}: {mode}", **st)
        e = s[(s["model"] == model) & (s["mode"] == "exact")]["mae_D"].mean()
        ax.axhline(e, color=st["color"], linewidth=0.8, linestyle=ls, alpha=0.5)
    ax.set(xscale="log", yscale="log", xlabel="shots per circuit S", title=f"{ev}, N = {NMAX} (thin lines: exact simulation)")
axes[0].set_ylabel("MAE (D), mean ± sd (log scale)")
axes[1].legend(fontsize=7.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "explore07_shots.png")
plt.show()

## 4. Hardware noise (FakeFez noise model on Aer)

A subset, because each noisy 10-qubit circuit takes about a second to simulate: training set S_(0,100), and
`N_TEST` development molecules. Four versions of the same predictions:
1. **exact**;
2. **shots only** (1,000 shots, ideal device);
3. **noisy**: every test-kernel entry from the transpiled overlap circuit on the noisy simulator, 1,000 shots;
4. **noisy, mitigated**: one extra circuit per molecule (its overlap with itself, exactly 1 without noise) estimates
   how much of the state survives, and each row is rescaled under a global-depolarizing assumption
   (`noise.depolarizing_mitigation`).
The training kernel is exact in this part (PLAN §7.6: a noisy training kernel at N = 100 needs 4,950 more circuits);
the projected kernel and the team's ⟨Z⟩ ridge run **end to end** noisy (training and test circuits), which is cheap
for them. The subset is small: read the differences as indicative.

In [ ]:
NOISE_N, N_TEST = (40, 8) if SMOKE else (100, 30)
SHOTS_NOISE = 1000
ids_n = splits.train[0][100][:NOISE_N]  # S_(0,100) (its first NOISE_N molecules in smoke runs)
rng = np.random.default_rng(2027)
test_ids = rng.choice(splits.dev, size=N_TEST, replace=False)
y_test = y.loc[test_ids].to_numpy()
N_PARAMS = NOISE_N if NOISE_N in SIZES else NMAX  # whose chosen settings to use (smoke runs: the N = 100 ones)
fitn = fixed_fit(FID, 0, NOISE_N) if NOISE_N in SIZES else None
if fitn is None:
    p, _ = chosen(FID, 0, N_PARAMS)
    enc, reps = KERNELS[FID]
    fitn = KernelRidgeFitter(XA, FidelityKernel(enc, reps, gammas=[p["gamma"]]), 0, scaling=SC_B, target=TG_B,
                             reduction=RED, alphas=(p["alpha"],)).fit(ids_n, y.loc[ids_n])
g = fitn.embed_setting_["gamma"]
At = g * fitn.transform_inputs(XA.loc[test_ids].to_numpy())
Atr = g * fitn.transform_inputs(XA.loc[ids_n].to_numpy())
enc_circ = fitn.kernel.circuit(RED[1])
n_q = enc_circ.num_qubits
K_exact = fitn.gram_to_train(XA.loc[test_ids].to_numpy())
pairs_a, pairs_b = np.repeat(At, len(Atr), axis=0), np.tile(Atr, (len(At), 1))
start = time.perf_counter()
K_noisy = sample_overlaps(enc_circ, pairs_a, pairs_b, SHOTS_NOISE, backend=BACKEND, seed=11, n_jobs=-1, chunk=25).reshape(K_exact.shape)
survival = sample_overlaps(enc_circ, At, At, SHOTS_NOISE, backend=BACKEND, seed=12, n_jobs=-1, chunk=5)
print(f"{K_noisy.size + len(At):,} noisy overlap circuits in {(time.perf_counter() - start) / 60:.1f} min")
K_shots = shot_kernel(K_exact, SHOTS_NOISE, np.random.default_rng(13))
K_mit = depolarizing_mitigation(K_noisy, survival, n_q)
noisy_rows = []
for mode, K in (("exact", K_exact), ("shots only", K_shots), ("noisy", K_noisy), ("noisy, mitigated", K_mit)):
    noisy_rows.append({"model": FID, "mode": mode, **scores(y_test, fitn.predict_from_gram(K))})
print(f"survival probability of x with itself: mean {survival.mean():.3f} (1 without noise; 2^-{n_q} = {2.0**-n_q:.4f} if fully scrambled)")

In [ ]:
def noisy_features(encoder, angles_tr, angles_te, bases, seed):
    tr = sample_bloch(encoder, angles_tr, SHOTS_NOISE, backend=BACKEND, seed=seed, bases=bases, n_jobs=-1, chunk=10)
    te = sample_bloch(encoder, angles_te, SHOTS_NOISE, backend=BACKEND, seed=seed + 100, bases=bases, n_jobs=-1, chunk=10)
    return tr, te


start = time.perf_counter()
# Projected kernel, end to end
p, _ = chosen("pqk", 0, N_PARAMS)
fitpn = KernelRidgeFitter(XA, ProjectedKernel("zz", 2, gammas=[p["gamma"]], gammas_p=[p["gamma_p"]]), 0, scaling=SC_B,
                          target=TG_B, reduction=RED, alphas=(p["alpha"],)).fit(ids_n, y.loc[ids_n])
gp = fitpn.embed_setting_["gamma"]
enc_p = fitpn.kernel.circuit(RED[1])
Btr_n, Bte_n = noisy_features(enc_p, gp * fitpn.transform_inputs(XA.loc[ids_n].to_numpy()),
                              gp * fitpn.transform_inputs(XA.loc[test_ids].to_numpy()), "XYZ", 21)
Btr_x, Bte_x = fitpn.train_embedding_, fitpn.embed_X(XA.loc[test_ids].to_numpy())
ks = fitpn.kernel_setting_
r_shot = np.random.default_rng(22)
Btr_s, Bte_s = shot_expectations(Btr_x, SHOTS_NOISE, r_shot), shot_expectations(Bte_x, SHOTS_NOISE, r_shot)
for mode, (Btr_m, Bte_m) in (("exact", (Btr_x, Bte_x)), ("shots only", (Btr_s, Bte_s)), ("noisy", (Btr_n, Bte_n))):
    mdl = KernelRidge(kernel="precomputed", alpha=fitpn.alpha_).fit(fitpn.kernel.gram(Btr_m, Btr_m, ks), fitpn.z_train_)
    noisy_rows.append({"model": "pqk", "mode": mode, **scores(y_test, fitpn.predict_from_gram(fitpn.kernel.gram(Bte_m, Btr_m, ks), mdl))})

# The team's quantum-feature ridge, end to end (Z basis only)
p, _ = chosen("qridge", 0, N_PARAMS)
est, _ = quantum_ridge_build(0, SC_B, TG_B, RED, XA.shape[1])
est.set_params(**p).fit(XA.loc[ids_n].to_numpy(), y.loc[ids_n].to_numpy())
prefix, qr = est.regressor_[:-1], est.regressor_[-1]
scaler = qr.pipeline_[0]
gq = p["regressor__model__gamma"]
a_tr = gq * scaler.transform(prefix.transform(XA.loc[ids_n].to_numpy()))
a_te = gq * scaler.transform(prefix.transform(XA.loc[test_ids].to_numpy()))
enc_q = qr.pipeline_[1].circuit_
Ftr_n, Fte_n = noisy_features(enc_q, a_tr, a_te, "Z", 31)
Ftr_x, Fte_x = qr.quantum_features(prefix.transform(XA.loc[ids_n].to_numpy())), qr.quantum_features(prefix.transform(XA.loc[test_ids].to_numpy()))
z_tr = est.transformer_.transform(y.loc[ids_n].to_numpy()[:, None]).ravel()
r_shot = np.random.default_rng(32)
for mode, (Ftr_m, Fte_m) in (("exact", (Ftr_x, Fte_x)), ("shots only", (shot_expectations(Ftr_x, SHOTS_NOISE, r_shot), shot_expectations(Fte_x, SHOTS_NOISE, r_shot))),
                             ("noisy", (Ftr_n, Fte_n))):
    rdg = Ridge(alpha=p["regressor__model__alpha"]).fit(Ftr_m, z_tr)
    pred = clip_predictions(est.transformer_.inverse_transform(rdg.predict(Fte_m)[:, None]).ravel())
    noisy_rows.append({"model": "qridge", "mode": mode, **scores(y_test, pred)})
print(f"projected kernel and ⟨Z⟩ ridge, noisy end to end: {(time.perf_counter() - start) / 60:.1f} min")
noisy = pd.DataFrame(noisy_rows)
_ = save_result(noisy, "explore07_noise", backend=BACKEND, shots=SHOTS_NOISE, n_train=NOISE_N, n_test=N_TEST, seed=0,
                label="exploratory; FakeFez noise model on Aer (local); small development subset")
display(noisy.assign(model=lambda d: d["model"].map(MODEL_LABELS)).set_index(["model", "mode"])
        [["mae_D", "rmse_D", "r2", "bias_D"]].style.format("{:.3f}")
        .set_caption(f"{N_TEST} development molecules, trained on S_(0,{NOISE_N}): exact vs shots vs {BACKEND} noise"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
ax = axes[0]
ax.scatter(K_exact.ravel(), K_noisy.ravel(), s=6, color=MODEL_COLORS[FID], alpha=0.35, label="noisy (1,000 shots)")
ax.scatter(K_exact.ravel(), K_mit.ravel(), s=6, color=MODEL_COLORS["rbf_krr"], alpha=0.35, label="noisy, mitigated")
lim = [0, max(1e-3, K_exact.max(), K_noisy.max()) * 1.05]
ax.plot(lim, lim, color=AXIS, linewidth=1)
ax.axhline(2.0**-n_q, color=MUTED, linewidth=0.8, linestyle=":")
ax.set(xlim=lim, ylim=lim, xlabel="exact kernel value", ylabel="estimated kernel value",
       title=f"Fidelity-kernel entries on {BACKEND} noise ({n_q} qubits)")
ax.legend(fontsize=8)
ax = axes[1]
B_flat = np.c_[Bte_x.ravel(), Bte_n.ravel()]
ax.scatter(B_flat[:, 0], B_flat[:, 1], s=6, color=MODEL_COLORS["pqk"], alpha=0.35)
ax.plot([-1, 1], [-1, 1], color=AXIS, linewidth=1)
ax.set(xlabel="exact Bloch coordinate", ylabel="noisy estimate", title="Projected-kernel features under noise",
       xlim=(-1, 1), ylim=(-1, 1), aspect="equal")
fig.tight_layout()
fig.savefig(FIG_DIR / "explore07_noise.png")
plt.show()

## Summary

*(Written from this run's outputs; see the cells above.)*